# Cascading Steering Experiments

Colab workflow for computing steering vectors and running phase 1 experiments.

Primary model: `meta-llama/Meta-Llama-3.1-8B-Instruct`
Fallback model: `mistralai/Mistral-7B-Instruct-v0.3`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/cascading-steering-attacks')
STEERING_DIR = DRIVE_ROOT / 'steering_vectors'
RESULTS_DIR = DRIVE_ROOT / 'results'
STEERING_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('Steering vectors ->', STEERING_DIR)
print('Results ->', RESULTS_DIR)


In [ ]:
%cd /content
REPO_URL = 'https://github.com/your-user/MAS-Activation-Cascades.git'
REPO_DIR = Path('/content/MAS-Activation-Cascades')
if not REPO_DIR.exists():
    !git clone $REPO_URL
%cd /content/MAS-Activation-Cascades


In [ ]:
!bash scripts/setup_references.sh


In [ ]:
!pip install --upgrade pip
!pip uninstall -y torch torchvision torchaudio
!pip install --upgrade --force-reinstall torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -r requirements.txt
!pip install -e ./third_party/camel
!python scripts/check_setup.py


In [ ]:
from huggingface_hub import notebook_login
notebook_login()

# If you do not have Llama 3 access yet, switch MODEL_NAME to the fallback below.
MODEL_NAME = 'meta-llama/Meta-Llama-3.1-8B-Instruct'
# MODEL_NAME = 'mistralai/Mistral-7B-Instruct-v0.3'


In [ ]:
import torch
import transformers

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('Torch:', torch.__version__)
print('Transformers:', transformers.__version__)


In [ ]:
from src.backends.steering_backend import CleanModelBackend

backend = CleanModelBackend(
    model_name=MODEL_NAME,
    max_new_tokens=32,
    do_sample=False,
)
sample = backend.generate_from_messages([{'role': 'user', 'content': 'Say hello in one sentence.'}])
print(sample.response_text)
print('Prompt tokens:', sample.prompt_token_count)
print('Completion tokens:', sample.completion_token_count)


## Build TA2 Contrastive Pairs

Generate a contrastive-pair JSON file from the local TA2 harmful dataset clone before computing steering vectors.


In [ ]:
PAIRS_PATH = Path('data/contrastive_pairs/ta2_harmful_pairs.json')
STEERING_PATH = STEERING_DIR / 'harmfulness_llama3_8b.pt'
!python scripts/build_ta2_pairs.py --dataset harmful --output "$PAIRS_PATH"
!python src/steering/compute_vectors.py \
    --model "$MODEL_NAME" \
    --pairs-path "$PAIRS_PATH" \
    --output "$STEERING_PATH" \
    --device cuda \
    --dtype auto


## Smoke Test: Single-Agent Steering Validation


In [ ]:
!python experiments/run_phase1.py \
    --experiment 1.1 \
    --model "$MODEL_NAME" \
    --steering-vector "$STEERING_PATH" \
    --results-dir "$RESULTS_DIR" \
    --n-tasks 3


## Serve Clean-Agent Backend

Start an OpenAI-compatible vLLM server for the clean agents used in experiments 1.2 to 1.4. With `Meta-Llama-3.1-8B-Instruct`, this is realistically an A100-class path because the notebook also keeps a separate steered local model loaded. On a T4, switch `MODEL_NAME` to the fallback model above and treat the multi-agent runs as smoke tests.


In [ ]:
import subprocess
import time
from pathlib import Path
from urllib.error import URLError
from urllib.request import urlopen

CLEAN_API_BASE = 'http://127.0.0.1:8000/v1'
VLLM_LOG = Path('/content/vllm-clean.log')

if 'vllm_process' in globals() and vllm_process.poll() is None:
    print('vLLM already running at', CLEAN_API_BASE)
else:
    log_handle = VLLM_LOG.open('w')
    vllm_process = subprocess.Popen([
        'python', '-m', 'vllm.entrypoints.openai.api_server',
        '--model', MODEL_NAME,
        '--host', '127.0.0.1',
        '--port', '8000',
        '--gpu-memory-utilization', '0.85',
        '--max-model-len', '4096',
    ], stdout=log_handle, stderr=subprocess.STDOUT)
    for attempt in range(60):
        try:
            with urlopen(f'{CLEAN_API_BASE}/models', timeout=5) as response:
                print('vLLM ready:', response.status)
                break
        except URLError:
            time.sleep(5)
    else:
        tail = VLLM_LOG.read_text(encoding='utf-8', errors='ignore')[-4000:]
        raise RuntimeError('vLLM did not become ready. Recent log output:\n' + tail)


## Two-Agent Chain


In [ ]:
!python experiments/run_phase1.py \
    --experiment 1.2 \
    --model "$MODEL_NAME" \
    --steering-vector "$STEERING_PATH" \
    --steering-strength 1.0 \
    --results-dir "$RESULTS_DIR" \
    --clean-api-base "$CLEAN_API_BASE"


## Three-Agent Chain


In [ ]:
!python experiments/run_phase1.py \
    --experiment 1.3 \
    --model "$MODEL_NAME" \
    --steering-vector "$STEERING_PATH" \
    --steering-strength 1.0 \
    --results-dir "$RESULTS_DIR" \
    --clean-api-base "$CLEAN_API_BASE"


## Star Topology


In [ ]:
!python experiments/run_phase1.py \
    --experiment 1.4 \
    --model "$MODEL_NAME" \
    --steering-vector "$STEERING_PATH" \
    --steering-strength 1.0 \
    --results-dir "$RESULTS_DIR" \
    --clean-api-base "$CLEAN_API_BASE"


In [ ]:
print('Steering directory contents:')
!ls -lah $STEERING_DIR
print('Results directory contents:')
!find $RESULTS_DIR -maxdepth 2 -type f | sort
